In [7]:
from common_helper_functions import *
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import average_precision_score, make_scorer
from scipy.stats import loguniform

In [ ]:
def to_xy(features_df, label_df, id_col='gene_id', label_col='label'):
    if label_col not in label_df.columns:
        raise ValueError(f"Label column '{label_col}' not found in label_df.")
    df = features_df.merge(label_df[[id_col, label_col]], on=id_col, how='inner')
    y = df[label_col].values
    feat_cols = [c for c in df.columns if c not in [id_col, label_col] and np.issubdtype(df[c].dtype, np.number)]
    X = df[feat_cols].to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, y


def nested_xgb_auprc(X, y, outer_splits=5, inner_splits=5, n_iter=40, random_state=42):
    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
    scorer = make_scorer(average_precision_score, response_method="predict_proba")

    param_dist = {
        'reg_alpha': loguniform(1e-5, 1e-2),
        'reg_lambda': loguniform(1e-5, 1e-2),
    }

    fold_rows = []
    for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        base = XGBClassifier(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='logloss',
            tree_method='hist',
            n_jobs=-1,
            random_state=random_state,
        )

        search = RandomizedSearchCV(
            estimator=base,
            param_distributions=param_dist,
            n_iter=n_iter,
            scoring=scorer,
            cv=inner_cv,
            n_jobs=-1,
            random_state=random_state,
            verbose=0,
        )
        search.fit(X_tr, y_tr)

        best_model = search.best_estimator_
        y_prob = best_model.predict_proba(X_te)[:, 1]
        test_ap = average_precision_score(y_te, y_prob)

        fold_rows.append({
            'fold': fold,
            'test_auprc': test_ap,
            'reg_alpha': search.best_params_['reg_alpha'],
            'reg_lambda': search.best_params_['reg_lambda'],
            'inner_best_score': search.best_score_,
        })

    return pd.DataFrame(fold_rows)

In [ ]:
embedding_datasets = {
    'Omics': omics,
    'STRING': string,
    'STRING_EXP': string_exp,
    'Orthrus': orthrus,
    'Emogi_Predictions': emogi_predictions,
}

all_folds, summary = [], []

for name, df in embedding_datasets.items():
    X, y = to_xy(df, emogi)
    res = nested_xgb_auprc(X, y, outer_splits=5, inner_splits=5, n_iter=40, random_state=42)
    res['dataset'] = name
    all_folds.append(res)
    summary.append({
        'Embedding': name,
        'auPRC_mean': res['test_auprc'].mean(),
        'auPRC_std': res['test_auprc'].std(),
        'reg_alpha_median': res['reg_alpha'].median(),
        'reg_lambda_median': res['reg_lambda'].median(),
    })

fold_results = pd.concat(all_folds, ignore_index=True)
summary_df = pd.DataFrame(summary).sort_values('auPRC_mean', ascending=False)

display(summary_df)
display(fold_results.head())

os.makedirs("results/hyperparam_tuning", exist_ok=True)
summary_df.to_csv("results/hyperparam_tuning/xgb_nestedcv_summary.csv", index=False)
fold_results.to_csv("results/hyperparam_tuning/xgb_nestedcv_folds.csv", index=False)